# 05b Context CatBoost Baseline

Optional full Colab context-only baseline. This is stronger than the smoke `constant_mean` baseline and is the recommended tabular comparison once real Statcast rows exist.

Required inputs:

- `manifests/bbe_events_v1.parquet`
- `configs/targets/target_registry_v1.yaml`

Created outputs:

- `datasets/context_dataset_v1/manifest.parquet`
- `predictions/context_catboost_mlb_2024_2026_v2/predictions_v1.parquet`
- `predictions/context_catboost_mlb_2024_2026_v2/metrics_v1.json`

Next: run `13_full_sequence_baseline.ipynb`, `14_full_video_baseline.ipynb`, then set `SOURCE_RUNS` in `15_full_fusion.ipynb` to include `context_catboost_mlb_2024_2026_v2`.

In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
RUN_ID = run_id(RUN_PROFILE, 'context_run_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('RUN_ID =', RUN_ID)


## Install CatBoost

This cell is intentionally explicit. Set `INSTALL_CATBOOST=True` only when you want Colab to download the package.

In [ ]:
import subprocess
import sys

CONTEXT_SETTINGS = stage_settings(RUN_PROFILE, 'context_catboost')
INSTALL_CATBOOST = bool(CONTEXT_SETTINGS.get('install_default', True))

if INSTALL_CATBOOST:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'catboost'])
else:
    print('CatBoost install skipped. If import fails below, set INSTALL_CATBOOST=True and rerun this cell.')


## Input Check

In [ ]:
from sport_pipeline.artifact_check import check_artifacts

required = ['manifests/bbe_events_v1.parquet']
print(check_artifacts(BASE_DIR, required))

## Run CatBoost Context Baseline

In [ ]:
try:
    import catboost  # noqa: F401
except ImportError as exc:
    raise RuntimeError('CatBoost is not installed. Set INSTALL_CATBOOST=True above and rerun the install cell.') from exc

from sport_pipeline.train_context_baseline import run_context_baseline

CONTEXT_SETTINGS = stage_settings(RUN_PROFILE, 'context_catboost')
outputs = run_context_baseline(
    BASE_DIR,
    run_id=RUN_ID,
    model_family='catboost',
    catboost_task_type=str(CONTEXT_SETTINGS.get('task_type', 'CPU')),
    catboost_devices=CONTEXT_SETTINGS.get('devices'),
    resume=bool(CONTEXT_SETTINGS.get('resume', True)),
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())


## Output Check

In [ ]:
expected = [
    'datasets/context_dataset_v1/manifest.parquet',
    f'predictions/{RUN_ID}/predictions_v1.parquet',
    f'predictions/{RUN_ID}/metrics_v1.json',
]
print(check_artifacts(BASE_DIR, expected))
print('REPORT COMMAND:')
print(f'python -m sport_pipeline.reports.build_static --base-dir {BASE_DIR} --run-id {RUN_ID}')